Import libraries

In [1]:
import warnings

import cmle
import random
import numpy as np
import pandas as pd
import re
from sklearn.compose import make_column_transformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Set random seeds for reproducible results across all runs
seed = 42
random.seed(seed)
np.random.seed(seed)
warnings.filterwarnings('ignore')  # Suppress warning messages for cleaner output

Load Data:

In [2]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

import warnings

import cmle
import random
import numpy as np
import pandas as pd
import re
from sklearn.compose import make_column_transformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Set random seeds for reproducible results across all runs
seed = 42
random.seed(seed)
np.random.seed(seed)
warnings.filterwarnings('ignore')  # Suppress warning messages for cleaner outputData Analysis:

In [3]:
train_df.sort_index()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


Replace any "?" with nan values (on the off chance they weren't already replaced with nan)

In [4]:
train_df_nan = train_df.replace("?", np.nan)
test_df_nan = test_df.replace("?", np.nan)
train_df_nan.shape

(7613, 5)

In [5]:
def clean_text(text):
    text = str (text).lower()
    text = re.sub(r'https?://\S+|www\.\S+','',text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()



In [6]:
train_df_nan['text'] = train_df_nan['text'].apply(clean_text)
test_df_nan['text'] = test_df_nan['text'].apply(clean_text)
train_df_nan['location'] = train_df_nan['location'].apply(clean_text)
test_df_nan['location'] = test_df_nan['location'].apply(clean_text)
train_df_nan['keyword'] = train_df_nan['keyword'].apply(clean_text)
test_df_nan['keyword'] = test_df_nan['keyword'].apply(clean_text)

In [7]:
train_df_nan["combined_text"] = train_df_nan["keyword"] + " " + train_df_nan["text"]
test_df_nan["combined_text"] = test_df_nan["keyword"] + " " + test_df_nan["text"]

In [8]:
train_df_nan.to_csv('bob.csv')

In [9]:
train_df_nan.info()
train_df_nan.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             7613 non-null   int64 
 1   keyword        7613 non-null   object
 2   location       7613 non-null   object
 3   text           7613 non-null   object
 4   target         7613 non-null   int64 
 5   combined_text  7613 non-null   object
dtypes: int64(2), object(4)
memory usage: 357.0+ KB


,id,target
count,7613.000000,7613.00000
mean,5441.934848,0.42966
std,3137.116090,0.49506
min,1.000000,0.00000
25%,2734.000000,0.00000
50%,5408.000000,0.00000
75%,8146.000000,1.00000
max,10873.000000,1.00000


In [10]:
#X = train_df_nan['text']
#y = train_df_nan['target']
X = train_df_nan["combined_text"]
y = train_df_nan["target"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed)

In [11]:
#Logistics Regression
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

#Decision Tree Classifier
model2 = DecisionTreeClassifier()
model2.fit(X_train_tfidf, y_train)

#
model3 = KNeighborsClassifier()
model3.fit(X_train_tfidf, y_train)

KNeighborsClassifier()

In [12]:
#Logistics
y_val_pred = model.predict(X_val_tfidf)
accuracy = accuracy_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
print("Logistics Accuracy:", accuracy)
print("Logistics F1 score:", f1)

#Decision Tree Classifier
y_val_pred = model2.predict(X_val_tfidf)
accuracy = accuracy_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
print("Decision Tree Classifier Accuracy:", accuracy)
print("Decision Tree Classifier F1 score:", f1)

#Decision Tree Classifier
y_val_pred = model3.predict(X_val_tfidf)
accuracy = accuracy_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
print("Kneighbour Accuracy:", accuracy)
print("Kneighbour F1 score:", f1)

Logistics Accuracy: 0.7984241628365069
Logistics F1 score: 0.7473251028806585
Decision Tree Classifier Accuracy: 0.7019041365725541
Decision Tree Classifier F1 score: 0.6373801916932907
Kneighbour Accuracy: 0.7518056467498359
Kneighbour F1 score: 0.7078825347758888
